# 12 Runtime Adapter Checks

In [1]:
import pathlib
import sys

_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(_root))

from src.runtime.openai_agents_runtime import OpenAIAgentsRuntimeAdapter
from src.schemas.events import RuntimeEventType

In [2]:
adapter = OpenAIAgentsRuntimeAdapter()
handle = await adapter.start_session("sess_runtime_nb", {"agent_id": "runtime-nb"})
assert handle.session_id == "sess_runtime_nb"

health = await adapter.healthcheck()
caps = adapter.get_capabilities()
print("health:", health.state.value, health.reason)
print("capabilities provider:", caps.provider_id)

context = {
    "run_id": "run_runtime_nb",
    "job_id": "job_runtime_nb",
    "task_id": "task_runtime_nb",
    "agent_id": "agent_runtime_nb",
    "planned_tool_call": {
        "call_id": "tc_runtime_nb",
        "tool_name": "fake_tool",
        "arguments": {"x": 1},
        "risk_tier": "low",
        "is_state_changing": False,
    },
}
events = []
async for event in adapter.run_turn("sess_runtime_nb", "hello", context):
    events.append(event)
    print(event.event_type.value, event.payload)

assert any(e.event_type == RuntimeEventType.TOOL_INTENT for e in events)
print("PASS: runtime adapter planned tool-intent path")

health: healthy adapter-initialized
capabilities provider: openai
tool_intent {}
PASS: runtime adapter planned tool-intent path
